In [1]:
import pandas as pd

# Read in dataset from articles dataset
articles_df = pd.read_csv("../data/articles.csv")
articles_df

,match_id,source,url,text
0,1,espn,https://www.espn.co.uk/football/match/_/gameId...,50%\nPossession\n13\nShots on Goal\n4\n20\nSho...
1,1,bbc,http://news.bbc.co.uk/sport2/hi/football/eng_p...,| \n\t\t\t\t\n\t\t\t\tCarlos Tevez celebrates ...
2,1,skysports,https://www.skysports.com/football/manchester-...,Manchester United vs Birmingham City; Premier ...
3,1,espn,https://www.espn.co.uk/football/report/_/gameI...,Skip to main content\nSkip to navigation\nESPN...
4,2,espn,https://www.espn.co.uk/football/match/_/gameId...,53%\nPossession\n47%\n9\nShots on Goal\n7\n22\...
...,...,...,...,...
63,20,guardian,https://www.theguardian.com/football/live/2025...,Some reaction from Old Trafford:\nBryan Mbeumo...
64,21,espn,https://www.espn.co.uk/football/match/_/gameId...,Man United v Bournemouth\nMatch Timeline\nKOHT...
65,21,bbc,https://www.bbc.co.uk/sport/football/live/c4g6...,Goodnightpublished at 22:45 GMT 15 December 20...
66,21,guardian,https://www.theguardian.com/football/live/2025...,Match report\nI’ll leave you with Jamie Jackso...


In [2]:
# Function to extracts the text from the relevant articles in the dataset for a match

def get_match_articles(match_id):
    return articles_df[articles_df["match_id"]==match_id]

In [3]:
# Funtion that joins the text from all the different articles together

def build_context(match_articles):
    texts = match_articles["text"].tolist()
    return "\n\n".join(texts)

In [4]:
# Load in the cleaned dataset as a source of truth for many of the desired feature

matches_df = pd.read_csv("../data/clean_matches.csv")
matches_df

,match_id,date,ground,home team,home score,away team,away score,goalscorer(s)
0,1,2008-01-01,Old Trafford,Man United,1,Birmingham,0,['Carlos Tevez']
1,2,2008-11-01,Old Trafford,Man United,4,Hull,3,"['Cristiano Ronaldo (x2)', 'Nemanja Vidic', 'M..."
2,3,2016-01-19,Villa Park,Aston Villa,2,Wycombe,0,"['Cieran Clark', 'Idrissa Gana Gueye']"
3,4,2017-05-11,Old Trafford,Man United,1,Celta Vigo,1,"['Marounne Fellaini', 'Facundo Roncaglia']"
4,5,2017-12-14,Old Trafford,Man United,1,Bournemouth,0,['Romelu Lukaku']
5,6,2019-10-10,Old Trafford,Man United,3,Brighton,1,"['Andreas Pereira', 'Scott McTominay', 'Lewis ..."
6,7,2019-12-01,Old Trafford,Man United,2,Aston Villa,2,"['Jack Grealish', 'Tom Heaton (og)', 'Victor L..."
7,8,2019-12-14,King Power Stadium,Leicester,1,Norwich,1,"['Teemu Pukki', 'Tim Krul (og)']"
8,9,2020-02-24,Old Trafford,Man United,3,Watford,0,"['Bruno Fernandes', 'Antony Martial', 'Mason G..."
9,10,2021-12-11,Carrow Road,Norwich,0,Man United,1,['Cristiano Ronaldo']


In [5]:
def get_known_facts(match_id):
    row = matches_df[matches_df["match_id"] == match_id]
    row = row.iloc[0]
 
    return {
        "home_team": row["home team"],
        "away_team": row["away team"],
        "score": f'{row["home score"]}-{row["away score"]}',
        "ground": row["ground"],
        "known_goalscorers": row["goalscorer(s)"]
    }

In [6]:
# Function that builds the query that will be fed into the LLM

# Edit OUTPUT FORMAT when a decision has been made about what summary date we want

def build_query(context):
    return f"""
    You are a strict information extraction system operating on football match reports.
 
    KNOWN FACTS (already verified from the attendee's own records — do not re-derive these,
    use them only to disambiguate which match/players the text is referring to):
    - Home team: {known_facts['home_team']}
    - Away team: {known_facts['away_team']}
    - Final score: {known_facts['score']}
    - Ground: {known_facts['ground']}
    - Known goalscorers: {', '.join(known_facts['known_goalscorers']) or 'none recorded'}
 
    TASK:
    Extract ONLY the following, which are NOT already known:
    - The minute and scoring team for each goal
    - Any red cards
    - Home and away managers
    - Man of the match
    - Any stadium detail that adds to or conflicts with the known ground
 
    RULES:
    - Use ONLY information explicitly stated in the TEXT below.
    - Do NOT guess or infer missing data. If a field is not stated, leave it as an empty
      string or empty list.
    - Do NOT invent players, minutes, or names that are not present in the text.
    - Do NOT repeat or duplicate events.
    - Every fact you return MUST include a short verbatim quote (<=25 words, copied exactly
      from the TEXT) as its "evidence". If you cannot find a supporting quote, omit the fact.
 
    TEXT:
    {context}
    """.strip()

In [7]:
# Get API key for OpenRouter from .env file

import os
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")

In [8]:
# JSON format that we want output to take

RESPONSE_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "match_extraction",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "Home team": {"type": "string"},
                "Away team": {"type": "string"},
                "Final score": {"type": "string"},
                "Ground": {"type": "string"},
                "goals": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "minute": {"type": "string"},
                            "player": {"type": "string"},
                            "team": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["minute", "player", "team", "evidence"],
                        "additionalProperties": False,
                    },
                },
                "red_cards": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "minute": {"type": "string"},
                            "player": {"type": "string"},
                            "team": {"type": "string"},
                            "evidence": {"type": "string"},
                        },
                        "required": ["minute", "player", "team", "evidence"],
                        "additionalProperties": False,
                    },
                },
                "home_manager": {"type": "string"},
                "home_manager_evidence": {"type": "string"},
                "away_manager": {"type": "string"},
                "away_manager_evidence": {"type": "string"},
                "man_of_the_match": {"type": "string"},
                "man_of_the_match_evidence": {"type": "string"},
            },
            "required": [
                "Home team", "Away team", "Final score",
                "Ground", "goals", "red_cards",
                "home_manager", "home_manager_evidence",
                "away_manager", "away_manager_evidence",
                "man_of_the_match", "man_of_the_match_evidence",
            ],
            "additionalProperties": False,
        },
    },
}

In [9]:
# Python function that interacts with the LLM

# Reasons why OpenRouter was chosen as an API
# Access to multiple models (OpenAI, Google, HuggingFace) using one API
#       Flexibility identified as important at early stage of project
#       Avoids vendor lock in
# Inexpensive

# Reasons why openai/gpt-4o-mini was chosen as AI Model.
# Relaible: Widely supported across different APIs (crucial for early stages of project when archeticeture can change)
# Excellent for following strict output formats and JSON schema adherance
# Strong for long context reasoning
# Inexpensive: Works on cheap API tiers
# Fast enough for real time use (could get away with slower times since pipeline will only run infrequently)
# Note: not open source unlike Llama 3

import requests

def call_llm(query, retries=2):
    for attempt in range(retries + 1):
        response = requests.post(                                       # Send data to server
            url="https://openrouter.ai/api/v1/chat/completions",        # URL where OpenRouter recieves and sends responses
            headers={
                "Authorization": f"Bearer {API_KEY}",                   # Communicates API Key
                "Content-Type": "application/json",                     # Telling API we sending json data
            },
            json={
                "model": "openai/gpt-4o-mini",                          # The AI model we want to use
                "temperature": 0,                                       # temperature parameter determines the randomness of the models selection. Setting to 0 makes it as deterministic as possible. Recommended for consistent data extraction
                "max_tokens": 1200,                                     # Restricts the length of the response.... REQUIRED?
                "response_format": RESPONSE_SCHEMA,                     # Structure for the output enforced by the API
                "messages": [
                    {"role": "user", "content": query}
                ],
            },
            timeout=30,                                                 # Dictates the length of time the code will wait for the API to respond...... REQUIRED? TOO SHORT?
        )
        data = response.json()
 
        if "choices" not in data:
            if attempt == retries:
                return None, f"API error: {data}"
            time.sleep(1.5 * (attempt + 1))
            continue
 
        content = data["choices"][0]["message"]["content"]              #Output given by the AI. Defaults to the first response if the AI suggests multiple models
        try:
            return json.loads(content), None
        except json.JSONDecodeError as e:
            if attempt == retries:
                return None, f"JSON parse failed after {retries + 1} attempts: {e}"
            time.sleep(1)
 
    return None, "Unknown failure"

# TO DO: ADD ERROR PROCESSING DOWNSTREAM

In [10]:
# Composes the precedding functions to generate required summary for the sample matches

import json

output = []
for match_id,group in articles_df.groupby("match_id"):
    known_facts = get_known_facts(match_id)
    match_articles = get_match_articles(match_id)
    context = build_context(match_articles)
    query = build_query(context)
    LLM_ouput = call_llm(query)[0]
    output.append(LLM_ouput)

output

[{'Home team': 'Man United',
  'Away team': 'Birmingham',
  'Final score': '1-0',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '25',
    'player': 'Carlos Tevez',
    'team': 'Man United',
    'evidence': 'Tevez scored the only goal as Manchester United secured a narrow victory... Tevez... calmly slotted home the winner on 25 minutes.'}],
  'red_cards': [],
  'home_manager': 'Sir Alex Ferguson',
  'home_manager_evidence': 'United manager Sir Alex Ferguson, sat in the stands to serve out his two-match touchline ban...',
  'away_manager': 'Alex McLeish',
  'away_manager_evidence': 'Birmingham boss Alex McLeish - once a protege of Ferguson in their days together at Aberdeen...',
  'man_of_the_match': 'Carlos Tevez',
  'man_of_the_match_evidence': "BBC Sport Player Rater man of the match: Manchester United's Carlos Tevez 8.15"},
 {'Home team': 'Man United',
  'Away team': 'Hull',
  'Final score': '4-3',
  'Ground': 'Old Trafford',
  'goals': [{'minute': '3',
    'player': 'Cristiano 

In [11]:
# Saves output from LLM query into a json file

import json
with open("../data/match.json","w") as f:
    json.dump(output,f,indent=4)          # indent=4 means each indentation level in the json file is presented with 4 spaces for readibility